In [5]:
from google.colab import files

uploaded = files.upload()


Saving creditcard.csv to creditcard.csv


In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier

In [ ]:
df = pd.read_csv("creditcard.csv")

df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [ ]:
X = df.drop("Class", axis=1)
y = df["Class"]

In [ ]:
print(df.shape)

print(df.info())

print(df.isnull().sum())

print(df["Class"].value_counts())

df = df.drop_duplicates()

print(df.shape)

(284807, 31)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 2

In [ ]:
X = df.drop("Class", axis=1)
y = df["Class"]

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
lr_model = LogisticRegression(
    class_weight="balanced",
    random_state=42,
    max_iter=1000
)

lr_model.fit(X_train_scaled, y_train)

lr_pred = lr_model.predict(X_val_scaled)

lr_prob = lr_model.predict_proba(X_val_scaled)[:,1]

In [ ]:
lr_precision = precision_score(y_val, lr_pred)
lr_recall = recall_score(y_val, lr_pred)
lr_f1 = f1_score(y_val, lr_pred)
lr_roc = roc_auc_score(y_val, lr_prob)
lr_pr = average_precision_score(y_val, lr_prob)

print("===== Logistic Regression =====")
print("Precision:", lr_precision)
print("Recall:", lr_recall)
print("F1:", lr_f1)
print("ROC-AUC:", lr_roc)
print("PR-AUC:", lr_pr)

print(confusion_matrix(y_val, lr_pred))

print(classification_report(y_val, lr_pred))

===== Logistic Regression =====
Precision: 0.050683829444891394
Recall: 0.8873239436619719
F1: 0.0958904109589041
ROC-AUC: 0.9718722568891034
PR-AUC: 0.697147026274304
[[41308  1180]
 [    8    63]]
              precision    recall  f1-score   support

           0       1.00      0.97      0.99     42488
           1       0.05      0.89      0.10        71

    accuracy                           0.97     42559
   macro avg       0.53      0.93      0.54     42559
weighted avg       1.00      0.97      0.98     42559



In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_val)

rf_prob = rf_model.predict_proba(X_val)[:,1]

In [ ]:
rf_precision = precision_score(y_val, rf_pred)
rf_recall = recall_score(y_val, rf_pred)
rf_f1 = f1_score(y_val, rf_pred)
rf_roc_auc = roc_auc_score(y_val, rf_prob)
rf_pr_auc = average_precision_score(y_val, rf_prob)

print("===== Random Forest =====")
print("Precision:", rf_precision)
print("Recall:", rf_recall)
print("F1:", rf_f1)
print("ROC-AUC:", rf_roc_auc)
print("PR-AUC:", rf_pr_auc)

rf_cm = confusion_matrix(y_val, rf_pred)

print(rf_cm)

print(classification_report(y_val, rf_pred))

===== Random Forest =====
Precision: 1.0
Recall: 0.6901408450704225
F1: 0.8166666666666667
ROC-AUC: 0.9416537826090418
PR-AUC: 0.8581843036333372
[[42488     0]
 [   22    49]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     42488
           1       1.00      0.69      0.82        71

    accuracy                           1.00     42559
   macro avg       1.00      0.85      0.91     42559
weighted avg       1.00      1.00      1.00     42559



In [ ]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

scale_pos_weight = neg / pos

xgb_model = XGBClassifier(
    objective="binary:logistic",
    random_state=42,
    eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight,
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_val)

xgb_prob = xgb_model.predict_proba(X_val)[:,1]

In [ ]:
xgb_precision = precision_score(y_val, xgb_pred)
xgb_recall = recall_score(y_val, xgb_pred)
xgb_f1 = f1_score(y_val, xgb_pred)
xgb_roc = roc_auc_score(y_val, xgb_prob)
xgb_pr = average_precision_score(y_val, xgb_prob)

print("===== XGBoost =====")
print("Precision:", xgb_precision)
print("Recall:", xgb_recall)
print("F1:", xgb_f1)
print("ROC-AUC:", xgb_roc)
print("PR-AUC:", xgb_pr)

===== XGBoost =====
Precision: 0.9047619047619048
Recall: 0.8028169014084507
F1: 0.8507462686567164
ROC-AUC: 0.9761904604050589
PR-AUC: 0.8503228849822417


In [ ]:
results = pd.DataFrame({
    "Model":[
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    "Precision":[
        lr_precision,
        rf_precision,
        xgb_precision
    ],
    "Recall":[
        lr_recall,
        rf_recall,
        xgb_recall
    ],
    "F1":[
        lr_f1,
        rf_f1,
        xgb_f1
    ],
    "ROC-AUC":[
        lr_roc,
        rf_roc_auc,
        xgb_roc
    ],
    "PR-AUC":[
        lr_pr,
        rf_pr_auc,
        xgb_pr
    ]
})

results

,Model,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Logistic Regression,0.050684,0.887324,0.095890,0.971872,0.697147
1,Random Forest,1.000000,0.690141,0.816667,0.941654,0.858184
2,XGBoost,0.904762,0.802817,0.850746,0.976190,0.850323


In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

In [13]:
uploaded.keys()

dict_keys(['creditcard.csv'])

In [14]:
import pandas as pd

filename = next(iter(uploaded.keys()))
df = pd.read_csv(filename)

print(df.shape)
df.head()

(284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [16]:
X = df.drop("Class", axis=1)
y = df["Class"]

print(X.shape)
print(y.shape)

(284807, 30)
(284807,)


In [17]:
from sklearn.model_selection import train_test_split

# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# Second split: 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Training: (199364, 30)
Validation: (42721, 30)
Test: (42722, 30)


In [24]:
param_dist = {
    "n_estimators":[100,200,300,500],
    "max_depth":[None,10,20,30],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,2,4],
    "max_features":["sqrt","log2"],
    "bootstrap":[True,False]
}

rf = RandomForestClassifier(
    random_state=42,
    class_weight="balanced"
)

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=10,
    scoring="average_precision",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

random_search.fit(X_train,y_train)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


RandomizedSearchCV(cv=3,
                   estimator=RandomForestClassifier(class_weight='balanced',
                                                    random_state=42),
                   n_jobs=-1,
                   param_distributions={'bootstrap': [True, False],
                                        'max_depth': [None, 10, 20, 30],
                                        'max_features': ['sqrt', 'log2'],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [100, 200, 300, 500]},
                   random_state=42, scoring='average_precision', verbose=2)

In [25]:
print(random_search.best_params_)
print(random_search.best_score_)

best_rf = random_search.best_estimator_

{'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 30, 'bootstrap': True}
0.8532006903211865


In [27]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

In [28]:
best_pred = best_rf.predict(X_val)

best_prob = best_rf.predict_proba(X_val)[:,1]

print("Precision:", precision_score(y_val,best_pred))
print("Recall:", recall_score(y_val,best_pred))
print("F1:", f1_score(y_val,best_pred))
print("ROC-AUC:", roc_auc_score(y_val,best_prob))
print("PR-AUC:", average_precision_score(y_val,best_prob))

Precision: 0.9622641509433962
Recall: 0.6891891891891891
F1: 0.8031496062992126
ROC-AUC: 0.9535878763374248
PR-AUC: 0.814127881325955


In [29]:
test_pred = best_rf.predict(X_test)

test_prob = best_rf.predict_proba(X_test)[:,1]

print("===== TEST SET =====")

print("Precision:", precision_score(y_test,test_pred))
print("Recall:", recall_score(y_test,test_pred))
print("F1:", f1_score(y_test,test_pred))
print("ROC-AUC:", roc_auc_score(y_test,test_prob))
print("PR-AUC:", average_precision_score(y_test,test_prob))

===== TEST SET =====
Precision: 0.9636363636363636
Recall: 0.7162162162162162
F1: 0.8217054263565892
ROC-AUC: 0.9610032091742841
PR-AUC: 0.8266128122469952


In [31]:
import joblib

In [38]:
import joblib

joblib.dump(best_rf, "fraud_model.pkl")

print("Random Forest model saved successfully!")

Random Forest model saved successfully!


In [40]:
joblib.dump(best_rf, "fraud_model.pkl")
import joblib

joblib.dump(best_rf, "fraud_model.pkl")

print("Random Forest model saved successfully!")

print("Model saved successfully!")

Random Forest model saved successfully!
Model saved successfully!
